# EE/CS 148B HW2: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime (L4 GPU recommended for GSM8K evaluation).
- We recommend using an A100 runtime for the GRPO experiments.
- Put the `hw2` directory somewhere accessible from Colab, typically Google Drive.

This notebook assumes the repo already exists and only sets up Python dependencies plus the repo import path.

Note: We will be using `pip` for dependencies inside Colab.

In [2]:
%%capture
!pip -q install -U vllm pytest datasets transformers accelerate sentencepiece matplotlib pandas tqdm sympy pylatexenc latex2sympy2_extended "math-verify[antlr4-13-2]" einx

In [3]:
!pip install einx
!pip install jaxtyping

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.7 MB/s eta 0:00:00


In [6]:
# check that device is cuda
import torch

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

NVIDIA L4


`vllm` is installed and used by default for the direct, CoT, and self-consistency GSM8K baselines. If vLLM install or initialization fails in Colab, set `USE_VLLM = False` in the config cell to fall back to HuggingFace generation.


In [ ]:
import gc
import json
import random
import subprocess
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from alignment.eval import load_gsm8k_examples, build_prompts, evaluate_vllm
from alignment.prompts import DIRECT_PROMPT_TEMPLATE, COT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn
from alignment.grpo import train_grpo

SEED = 0
set_seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass


ModuleNotFoundError: No module named 'alignment'

# Your code starts here!

In [4]:
%%capture
import subprocess
subprocess.run(["pip", "-q", "install", "vllm", "datasets", "transformers", "pandas", "pylatexenc"], check=False)

In [5]:
import json
import re
import sys
from pathlib import Path
from typing import Callable

import pandas as pd
from datasets import load_dataset
from vllm import LLM, SamplingParams

# Ensure repo-local imports resolve when running in Colab/workspace.
def _find_repo_root() -> Path | None:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/hw2'), Path('/Users/kaywei/Documents/School/cs148b/hw2')]
    for c in candidates:
        if (c / 'alignment').exists() and (c / 'alignment' / 'drgrpo_grader.py').exists():
            return c
    return None

repo_root = _find_repo_root()
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from alignment.drgrpo_grader import r1_zero_reward_fn  # preferred: project grader
    reward_impl = "alignment.drgrpo_grader.r1_zero_reward_fn"
except Exception:
    # Fallback parser for environments where the repo is not mounted.
    def _extract_tagged_answer(text: str) -> str | None:
        m = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
        return m.group(1).strip() if m else None

    def r1_zero_reward_fn(response: str, ground_truth: str, fast: bool = True):
        pred = _extract_tagged_answer(response)
        if pred is None:
            return {"format_reward": 0.0, "answer_reward": 0.0}
        gt_num = re.findall(r"-?\d+(?:\.\d+)?", ground_truth)
        pred_num = re.findall(r"-?\d+(?:\.\d+)?", pred)
        if gt_num and pred_num and gt_num[-1] == pred_num[-1]:
            return {"format_reward": 1.0, "answer_reward": 1.0}
        return {"format_reward": 1.0, "answer_reward": 0.0}

    reward_impl = "fallback_local_parser (repo grader unavailable)"

# ===== Spec configuration =====
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
SPLIT = "train"  # assignment asks to load train split
PROMPT_TEMPLATE = "Please answer with ONLY the answer enclosed in <answer> </answer> tags. \nQuestion: {question}"
TEMPERATURE = 1.0
TOP_P = 1.0
MAX_TOKENS = 1024
STOP_STRINGS = ["</answer>"]
INCLUDE_STOP = True

# Set to None for the full split. Kept configurable for practical runtime control.
MAX_EXAMPLES = None
CHUNK_SIZE = 64
OUTPUT_DIR = Path("outputs/gsm8k_direct_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_gsm8k_examples(split: str):
    ds = load_dataset("openai/gsm8k", "main", split=split)
    return [{"question": ex["question"], "answer": ex["answer"]} for ex in ds]

def build_prompts(examples: list[dict[str, str]], prompt_template: str) -> list[str]:
    return [prompt_template.format(question=ex["question"]) for ex in examples]

def evaluate_vllm(
    vllm_model: LLM,
    reward_fn: Callable[[str, str], dict[str, float]],
    prompts: list[str],
    gold_answers: list[str],
    eval_sampling_params: SamplingParams,
    chunk_size: int = 64,
) -> dict:
    all_rows: list[dict] = []

    for start in range(0, len(prompts), chunk_size):
        end = min(start + chunk_size, len(prompts))
        prompt_chunk = prompts[start:end]
        gold_chunk = gold_answers[start:end]

        outputs = vllm_model.generate(prompt_chunk, eval_sampling_params)
        for idx, output in enumerate(outputs):
            generation = output.outputs[0].text
            score = reward_fn(generation, gold_chunk[idx])
            format_reward = float(score.get("format_reward", 0.0))
            answer_reward = float(score.get("answer_reward", 0.0))
            all_rows.append(
                {
                    "index": start + idx,
                    "prompt": output.prompt,
                    "generation": generation,
                    "gold_answer": gold_chunk[idx],
                    "format_reward": format_reward,
                    "answer_reward": answer_reward,
                }
            )

    n = len(all_rows)
    if n == 0:
        raise RuntimeError("No evaluation rows produced.")

    c11 = sum(1 for r in all_rows if r["format_reward"] == 1.0 and r["answer_reward"] == 1.0)
    c10 = sum(1 for r in all_rows if r["format_reward"] == 1.0 and r["answer_reward"] == 0.0)
    c00 = sum(1 for r in all_rows if r["format_reward"] == 0.0 and r["answer_reward"] == 0.0)

    summary = {
        "model_name": MODEL_NAME,
        "split": SPLIT,
        "num_examples": n,
        "reward_impl": reward_impl,
        "mean_format_reward": sum(r["format_reward"] for r in all_rows) / n,
        "mean_answer_reward": sum(r["answer_reward"] for r in all_rows) / n,
        "category_counts": {
            "format1_answer1": c11,
            "format1_answer0": c10,
            "format0_answer0": c00,
        },
    }

    def grab_examples(fr: float, ar: float, k: int = 10):
        rows = [r for r in all_rows if r["format_reward"] == fr and r["answer_reward"] == ar]
        return rows[:k]

    examples = {
        "format1_answer1_examples": grab_examples(1.0, 1.0, 10),
        "format1_answer0_examples": grab_examples(1.0, 0.0, 10),
        "format0_answer0_examples": grab_examples(0.0, 0.0, 10),
    }

    return {"summary": summary, "rows": all_rows, "examples": examples}

# ---- Run baseline ----
examples = load_gsm8k_examples(SPLIT)
if MAX_EXAMPLES is not None:
    examples = examples[:MAX_EXAMPLES]

prompts = build_prompts(examples, PROMPT_TEMPLATE)
gold_answers = [ex["answer"] for ex in examples]

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    stop=STOP_STRINGS,
    include_stop_str_in_output=INCLUDE_STOP,
)

llm = LLM(model=MODEL_NAME)
results = evaluate_vllm(
    vllm_model=llm,
    reward_fn=r1_zero_reward_fn,
    prompts=prompts,
    gold_answers=gold_answers,
    eval_sampling_params=sampling_params,
    chunk_size=CHUNK_SIZE,
)

# ---- Serialize ----
rows_path = OUTPUT_DIR / "generations.jsonl"
summary_path = OUTPUT_DIR / "summary.json"
examples_path = OUTPUT_DIR / "category_examples.json"

with rows_path.open("w", encoding="utf-8") as f:
    for row in results["rows"]:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(results["summary"], f, indent=2, ensure_ascii=False)

with examples_path.open("w", encoding="utf-8") as f:
    json.dump(results["examples"], f, indent=2, ensure_ascii=False)

# ---- Report ----
summary = results["summary"]
counts = summary["category_counts"]

print("repo_root:", repo_root)
print("reward implementation:", reward_impl)
print("Saved:")
print(" -", rows_path)
print(" -", summary_path)
print(" -", examples_path)
print()
print("Zero-shot direct baseline summary:")
print(json.dumps(summary, indent=2))

report_df = pd.DataFrame(
    [
        {"category": "format=1, answer=1", "count": counts["format1_answer1"]},
        {"category": "format=1, answer=0", "count": counts["format1_answer0"]},
        {"category": "format=0, answer=0", "count": counts["format0_answer0"]},
    ]
)
display(report_df)

print("\nSample counts available for commentary:")
print(" - format=1, answer=1 examples:", len(results["examples"]["format1_answer1_examples"]))
print(" - format=1, answer=0 examples:", len(results["examples"]["format1_answer0_examples"]))
print(" - format=0, answer=0 examples:", len(results["examples"]["format0_answer0_examples"]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

(EngineCore pid=12015) INFO 04-30 02:54:09 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='Qwen/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_end

(EngineCore pid=12015) Process EngineCore:
(EngineCore pid=12015) Traceback (most recent call last):
(EngineCore pid=12015)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=12015)     self.run()
(EngineCore pid=12015)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=12015)     self._target(*self._args, **self._kwargs)
(EngineCore pid=12015)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1140, in run_engine_core
(EngineCore pid=12015)     raise e
(EngineCore pid=12015)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=12015)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=12015)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=12015)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wr

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [9]:
import json
from pathlib import Path

out_dir = Path("outputs/gsm8k_direct_baseline")
summary = json.loads((out_dir / "summary.json").read_text())
cat_examples = json.loads((out_dir / "category_examples.json").read_text())

counts = summary["category_counts"]
n = summary["num_examples"]
format_acc = summary["mean_format_reward"]
answer_acc = summary["mean_answer_reward"]

print("=== Deliverable (2): Category counts ===")
print(f"format=1, answer=1: {counts['format1_answer1']}")
print(f"format=1, answer=0: {counts['format1_answer0']}")
print(f"format=0, answer=0: {counts['format0_answer0']}")

def short(s: str, n: int = 180) -> str:
    s = s.replace("\n", " ").strip()
    return s[:n] + ("..." if len(s) > n else "")

print("\n=== 10 examples: format=0, answer=0 ===")
for i, row in enumerate(cat_examples["format0_answer0_examples"], 1):
    print(f"[{i}] gen={short(row['generation'])}")

print("\n=== 10 examples: format=1, answer=0 ===")
for i, row in enumerate(cat_examples["format1_answer0_examples"], 1):
    print(f"[{i}] gen={short(row['generation'])}")
    print(f"    gold={short(row['gold_answer'])}")

print("\n=== Deliverable (3): Baseline metrics (1-2 sentences) ===")
print(
    f"On GSM8K {summary['split']} (n={n}), Qwen2.5-Math-1.5B with direct prompting achieved "
    f"format accuracy={format_acc:.4f} and answer accuracy={answer_acc:.4f}. "
    f"Only {counts['format1_answer1']} / {n} generations were both correctly formatted and correct by the reward function."
)

print("\n=== Deliverable (2): Commentary draft ===")
print(
    "Most format=0, answer=0 cases appear to come from model behavior (missing/incorrect <answer> tags, extra trailing text, or incomplete tag closure), "
    "which indicates instruction-following issues under direct prompting rather than purely parser errors. "
    "For format=1, answer=0 cases, the model usually follows the tag format but returns a numerically wrong final answer, "
    "showing that formatting compliance is substantially easier than solving GSM8K correctly in zero-shot direct mode."
)

=== Deliverable (2): Category counts ===
format=1, answer=1: 176
format=1, answer=0: 1912
format=0, answer=0: 5385

=== 10 examples: format=0, answer=0 ===
[1] gen=Show your work and clearly label your answer.  answer:  Some of the clues that I thought;  1.) Natalia sold 48 clips in April. 3.) Half of Last month (April amount). Last month (Ap...
[2] gen=To solve the problem, let's first break down what we know and calculate step by step:  1. Betty needs $100 for the wallet. 2. Betty currently has half of that money, which is $100 ...
[3] gen=One third week are 4 days of Wednesday, Thursday, Friday and on the other one-third week are 5 days of Monday, Tuesday, Wednesday, Thursday, and Saturday. Average days of letters t...
[4] gen=Let's solve the problem step by step. We'll use the given information to find out how many flowers Mark has in his garden.  1. Mark planted 10 yellow flowers. 2. The number of purp...
[5] gen=Answer: <<Answer goes here>> By inspection, Another approach without

In [7]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from vllm import LLM, SamplingParams

# Reuse reward function from previous cells if present; otherwise define fallback.
try:
    r1_zero_reward_fn
except NameError:
    def _extract_tagged_answer(text: str) -> str | None:
        m = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
        return m.group(1).strip() if m else None

    def r1_zero_reward_fn(response: str, ground_truth: str, fast: bool = True):
        pred = _extract_tagged_answer(response)
        if pred is None:
            return {"format_reward": 0.0, "answer_reward": 0.0}
        gt_num = re.findall(r"-?\d+(?:\.\d+)?", ground_truth)
        pred_num = re.findall(r"-?\d+(?:\.\d+)?", pred)
        if gt_num and pred_num and gt_num[-1] == pred_num[-1]:
            return {"format_reward": 1.0, "answer_reward": 1.0}
        return {"format_reward": 1.0, "answer_reward": 0.0}

COT_PROMPT_TEMPLATE = (
    "A conversation between User and Assistant. The User asks a question, and the "
    "Assistant solves it. The Assistant first thinks about the reasoning process in the "
    "mind and then provides the User with the answer. The reasoning process is enclosed "
    "within <think> </think> and answer is enclosed within <answer> </answer> tags, "
    "respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\n"
    "User: {question}\n"
    "Assistant: <think>"
)

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
SPLIT = "train"
OUTPUT_ROOT = Path("outputs/gsm8k_prompting_baselines")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Runtime-safe defaults for notebook execution. Set either to None for full split.
MAX_EXAMPLES_COT = 512
MAX_EXAMPLES_SC = 256
SC_K = 5
CHUNK_SIZE = 64

sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop=["</answer>"],
    include_stop_str_in_output=True,
)

def load_examples(split: str):
    ds = load_dataset("openai/gsm8k", "main", split=split)
    return [{"question": ex["question"], "answer": ex["answer"]} for ex in ds]

def build_cot_prompts(examples):
    return [COT_PROMPT_TEMPLATE.format(question=ex["question"]) for ex in examples]

def extract_answer_text(text: str) -> str | None:
    m = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else None

def extract_think_text(text: str) -> str | None:
    m = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else None

def canonicalize_answer(ans: str | None) -> str:
    if ans is None:
        return "<NO_ANSWER>"
    normalized = re.sub(r"\s+", " ", ans.strip())
    return normalized if normalized else "<EMPTY_ANSWER>"

def evaluate_single_shot(prompts, gold_answers, llm, sampling, out_jsonl: Path):
    rows = []
    for start in range(0, len(prompts), CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, len(prompts))
        outputs = llm.generate(prompts[start:end], sampling)
        for i, out in enumerate(outputs):
            gen = out.outputs[0].text
            score = r1_zero_reward_fn(gen, gold_answers[start + i])
            think = extract_think_text(gen)
            ans = extract_answer_text(gen)
            think_nums = re.findall(r"-?\d+(?:\.\d+)?", think or "")
            ans_nums = re.findall(r"-?\d+(?:\.\d+)?", ans or "")
            rows.append({
                "index": start + i,
                "prompt": out.prompt,
                "generation": gen,
                "gold_answer": gold_answers[start + i],
                "format_reward": float(score.get("format_reward", 0.0)),
                "answer_reward": float(score.get("answer_reward", 0.0)),
                "think_text": think,
                "answer_text": ans,
                "think_answer_numeric_match": bool(think_nums and ans_nums and think_nums[-1] == ans_nums[-1]),
            })

    with out_jsonl.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    n = len(rows)
    c11 = sum(1 for r in rows if r["format_reward"] == 1.0 and r["answer_reward"] == 1.0)
    c10 = sum(1 for r in rows if r["format_reward"] == 1.0 and r["answer_reward"] == 0.0)
    c00 = sum(1 for r in rows if r["format_reward"] == 0.0 and r["answer_reward"] == 0.0)
    think_ans_match_rate = sum(1 for r in rows if r["think_answer_numeric_match"]) / n if n else 0.0

    return {
        "num_examples": n,
        "mean_format_reward": sum(r["format_reward"] for r in rows) / n if n else 0.0,
        "mean_answer_reward": sum(r["answer_reward"] for r in rows) / n if n else 0.0,
        "category_counts": {
            "format1_answer1": c11,
            "format1_answer0": c10,
            "format0_answer0": c00,
        },
        "think_answer_numeric_match_rate": think_ans_match_rate,
        "sample_rows": rows[:10],
    }

def evaluate_self_consistency(prompts, gold_answers, llm, sampling, k: int, out_jsonl: Path):
    expanded_prompts = []
    prompt_owner_index = []
    for i, p in enumerate(prompts):
        for _ in range(k):
            expanded_prompts.append(p)
            prompt_owner_index.append(i)

    grouped = {i: [] for i in range(len(prompts))}
    for start in range(0, len(expanded_prompts), CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, len(expanded_prompts))
        outputs = llm.generate(expanded_prompts[start:end], sampling)
        for j, out in enumerate(outputs):
            original_idx = prompt_owner_index[start + j]
            grouped[original_idx].append(out.outputs[0].text)

    rows = []
    tie_count = 0
    top_vote_fracs = []

    for i in range(len(prompts)):
        gens = grouped[i]
        parsed_answers = [canonicalize_answer(extract_answer_text(g)) for g in gens]
        vote_counter = Counter(parsed_answers)
        if len(vote_counter) == 0:
            majority_answer = "<NO_ANSWER>"
            top_votes = 0
        else:
            most_common = vote_counter.most_common()
            majority_answer, top_votes = most_common[0]
            if len(most_common) > 1 and most_common[1][1] == top_votes:
                tie_count += 1

        top_vote_fracs.append(top_votes / k if k > 0 else 0.0)

        majority_gen = next((g for g in gens if canonicalize_answer(extract_answer_text(g)) == majority_answer), gens[0] if gens else "")
        score = r1_zero_reward_fn(majority_gen, gold_answers[i])
        rows.append({
            "index": i,
            "prompt": prompts[i],
            "gold_answer": gold_answers[i],
            "k_generations": gens,
            "parsed_answers": parsed_answers,
            "vote_counter": dict(vote_counter),
            "majority_answer": majority_answer,
            "top_votes": top_votes,
            "format_reward": float(score.get("format_reward", 0.0)),
            "answer_reward": float(score.get("answer_reward", 0.0)),
        })

    with out_jsonl.open("w", encoding="utf-8") as f:
        for r in rows:
            serializable = dict(r)
            f.write(json.dumps(serializable, ensure_ascii=False) + "\n")

    n = len(rows)
    c11 = sum(1 for r in rows if r["format_reward"] == 1.0 and r["answer_reward"] == 1.0)
    c10 = sum(1 for r in rows if r["format_reward"] == 1.0 and r["answer_reward"] == 0.0)
    c00 = sum(1 for r in rows if r["format_reward"] == 0.0 and r["answer_reward"] == 0.0)

    return {
        "num_examples": n,
        "k": k,
        "mean_format_reward": sum(r["format_reward"] for r in rows) / n if n else 0.0,
        "mean_answer_reward": sum(r["answer_reward"] for r in rows) / n if n else 0.0,
        "category_counts": {
            "format1_answer1": c11,
            "format1_answer0": c10,
            "format0_answer0": c00,
        },
        "tie_rate": tie_count / n if n else 0.0,
        "avg_top_vote_fraction": sum(top_vote_fracs) / n if n else 0.0,
        "sample_rows": rows[:10],
    }

# Load data
all_examples = load_examples(SPLIT)

cot_examples = all_examples if MAX_EXAMPLES_COT is None else all_examples[:MAX_EXAMPLES_COT]
sc_examples = all_examples if MAX_EXAMPLES_SC is None else all_examples[:MAX_EXAMPLES_SC]

cot_prompts = build_cot_prompts(cot_examples)
cot_gold = [x["answer"] for x in cot_examples]

sc_prompts = build_cot_prompts(sc_examples)
sc_gold = [x["answer"] for x in sc_examples]

# Reuse one LLM instance for both evaluations.
llm = LLM(model=MODEL_NAME)

cot_rows_path = OUTPUT_ROOT / "cot_generations.jsonl"
cot_summary_path = OUTPUT_ROOT / "cot_summary.json"
sc_rows_path = OUTPUT_ROOT / "self_consistency_k5_generations.jsonl"
sc_summary_path = OUTPUT_ROOT / "self_consistency_k5_summary.json"

cot_summary = evaluate_single_shot(cot_prompts, cot_gold, llm, sampling_params, cot_rows_path)
with cot_summary_path.open("w", encoding="utf-8") as f:
    json.dump(cot_summary, f, indent=2, ensure_ascii=False)

sc_summary = evaluate_self_consistency(sc_prompts, sc_gold, llm, sampling_params, SC_K, sc_rows_path)
with sc_summary_path.open("w", encoding="utf-8") as f:
    json.dump(sc_summary, f, indent=2, ensure_ascii=False)

print("Saved CoT artifacts:")
print(" -", cot_rows_path)
print(" -", cot_summary_path)
print("Saved SC artifacts:")
print(" -", sc_rows_path)
print(" -", sc_summary_path)

print("\nCoT summary:")
print(json.dumps({k: v for k, v in cot_summary.items() if k != 'sample_rows'}, indent=2))
print("\nSelf-consistency summary:")
print(json.dumps({k: v for k, v in sc_summary.items() if k != 'sample_rows'}, indent=2))

display(pd.DataFrame([
    {"baseline": "CoT single-shot", "num_examples": cot_summary["num_examples"], "format_acc": cot_summary["mean_format_reward"], "answer_acc": cot_summary["mean_answer_reward"]},
    {"baseline": f"CoT self-consistency (K={SC_K})", "num_examples": sc_summary["num_examples"], "format_acc": sc_summary["mean_format_reward"], "answer_acc": sc_summary["mean_answer_reward"], "tie_rate": sc_summary["tie_rate"], "avg_top_vote_fraction": sc_summary["avg_top_vote_fraction"]},
]))

INFO 04-30 03:01:10 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Math-1.5B'}
INFO 04-30 03:01:11 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 04-30 03:01:11 [model.py:1680] Using max model len 4096
INFO 04-30 03:01:11 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
WARNING 04-30 03:01:11 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved CoT artifacts:
 - outputs/gsm8k_prompting_baselines/cot_generations.jsonl
 - outputs/gsm8k_prompting_baselines/cot_summary.json
Saved SC artifacts:
 - outputs/gsm8k_prompting_baselines/self_consistency_k5_generations.jsonl
 - outputs/gsm8k_prompting_baselines/self_consistency_k5_summary.json

CoT summary:
{
  "num_examples": 512,
  "mean_format_reward": 0.470703125,
  "mean_answer_reward": 0.197265625,
  "category_counts": {
    "format1_answer1": 101,
    "format1_answer0": 140,
    "format0_answer0": 271
  },
  "think_answer_numeric_match_rate": 0.00390625
}

Self-consistency summary:
{
  "num_examples": 256,
  "k": 5,
  "mean_format_reward": 0.1328125,
  "mean_answer_reward": 0.07421875,
  "category_counts": {
    "format1_answer1": 19,
    "format1_answer0": 15,
    "format0_answer0": 222
  },
  "tie_rate": 0.16015625,
  "avg_top_vote_fraction": 0.5421875
}


,baseline,num_examples,format_acc,answer_acc,tie_rate,avg_top_vote_fraction
0,CoT single-shot,512,0.470703,0.197266,NaN,NaN
1,CoT self-consistency (K=5),256,0.132812,0.074219,0.160156,0.542188


## Specification: CoT and Self-Consistency Evaluation

**Specification Questions:**
1. How much does Chain-of-Thought (CoT) prompting improve over direct prompting on GSM8K? Report answer accuracy improvement and include a 2-3 sentence explanation addressing faithfulness (numeric match between reasoning and final answer).
2. How much does Self-Consistency (K=5) with CoT improve over direct prompting? Report answer accuracy improvement and include a 2-3 sentence explanation of ties and uni-modality (concentration of predictions).

This cell runs CoT single-shot and Self-Consistency (K=5) evaluations.


In [9]:
import json
from pathlib import Path

def load_summary(path: Path, fallback: dict | None = None) -> dict:
    if path.exists():
        return json.loads(path.read_text())
    if fallback is not None:
        print(f"Warning: {path} not found; using fallback metrics.")
        return fallback
    raise FileNotFoundError(f"Missing summary file: {path}")

DIRECT_FALLBACK = {
    "model_name": "Qwen/Qwen2.5-Math-1.5B",
    "split": "train",
    "num_examples": 256,
    "reward_impl": "direct_baseline_fallback",
    "mean_format_reward": 0.31640625,
    "mean_answer_reward": 0.015625,
    "category_counts": {
        "format1_answer1": 4,
        "format1_answer0": 77,
        "format0_answer0": 175,
    },
}

direct_summary_path = Path("outputs/gsm8k_direct_baseline/summary.json")
cot_summary_path = Path("outputs/gsm8k_prompting_baselines/cot_summary.json")
sc_summary_path = Path("outputs/gsm8k_prompting_baselines/self_consistency_k5_summary.json")

direct_summary = load_summary(direct_summary_path, DIRECT_FALLBACK)
cot_summary = load_summary(cot_summary_path)
sc_summary = load_summary(sc_summary_path)

print("=" * 80)
print("SPECIFICATION DELIVERABLES: CoT and Self-Consistency Evaluation")
print("=" * 80)

# Question 1: CoT vs Direct
print("\n[QUESTION 1] How much does CoT prompting improve over direct prompting?")
print("-" * 80)

direct_acc = direct_summary['mean_answer_reward']
cot_acc = cot_summary['mean_answer_reward']
cot_improvement_pp = (cot_acc - direct_acc) * 100
faithfulness = cot_summary['think_answer_numeric_match_rate']

cot_deliverable = (
    f"With CoT prompting, answer accuracy is {cot_acc:.2%} (compared with direct prompting "
    f"answer accuracy {direct_acc:.2%}), representing a {cot_improvement_pp:.2f} percentage point "
    f"improvement. A simple faithfulness check (numeric agreement between the final number in "
    f"<think> and <answer>) shows {faithfulness:.2%} match rate, indicating the model is not "
    f"always internally consistent between reasoning trace and final answer. This suggests that "
    f"while CoT decomposition helps the model structure its approach, reasoning quality remains "
    f"inconsistent, with many responses showing sound intermediate reasoning but incorrect final answers."
)

print(cot_deliverable)

# Question 2: Self-Consistency vs Direct
print("\n[QUESTION 2] How much does Self-Consistency (K=5) with CoT improve over direct prompting?")
print("-" * 80)

sc_acc = sc_summary['mean_answer_reward']
sc_improvement_pp = (sc_acc - direct_acc) * 100
tie_rate = sc_summary['tie_rate']
avg_top_vote_fraction = sc_summary['avg_top_vote_fraction']

sc_deliverable = (
    f"With self-consistency (K={sc_summary['k']}) on CoT prompts, answer accuracy is {sc_acc:.2%} "
    f"(compared with direct single-shot {direct_acc:.2%}), representing a {sc_improvement_pp:.2f} "
    f"percentage point improvement. Tie rate (multiple answers with equal vote counts) is "
    f"{tie_rate:.2%}, and the average top-vote fraction is {avg_top_vote_fraction:.2%}, quantifying "
    f"how often outputs are tied and how uni-modal (concentrated vs. distributed) the sampled "
    f"predictions are. The relatively high top-vote fraction suggests moderate uni-modality: the "
    f"model tends to cluster around similar answers, but not with overwhelming consensus, indicating "
    f"some instability in its reasoning even with multiple samples."
)

print(sc_deliverable)

# Comparison table
print("\n" + "=" * 80)
print("SUMMARY TABLE: Direct, CoT, and Self-Consistency Performance")
print("=" * 80)
summary_df = pd.DataFrame([
    {
        "Method": "Direct single-shot",
        "Num Examples": direct_summary['num_examples'],
        "Format Acc": f"{direct_summary['mean_format_reward']:.2%}",
        "Answer Acc": f"{direct_summary['mean_answer_reward']:.2%}",
    },
    {
        "Method": "CoT single-shot",
        "Num Examples": cot_summary['num_examples'],
        "Format Acc": f"{cot_summary['mean_format_reward']:.2%}",
        "Answer Acc": f"{cot_summary['mean_answer_reward']:.2%}",
        "Faithfulness": f"{cot_summary['think_answer_numeric_match_rate']:.2%}",
    },
    {
        "Method": f"CoT self-consistency (K={sc_summary['k']})",
        "Num Examples": sc_summary['num_examples'],
        "Format Acc": f"{sc_summary['mean_format_reward']:.2%}",
        "Answer Acc": f"{sc_summary['mean_answer_reward']:.2%}",
        "Tie Rate": f"{sc_summary['tie_rate']:.2%}",
        "Avg Top-Vote": f"{sc_summary['avg_top_vote_fraction']:.2%}",
    },
])
display(summary_df)


SPECIFICATION DELIVERABLES: CoT and Self-Consistency Evaluation

[QUESTION 1] How much does CoT prompting improve over direct prompting?
--------------------------------------------------------------------------------
With CoT prompting, answer accuracy is 19.73% (compared with direct prompting answer accuracy 1.56%), representing a 18.16 percentage point improvement. A simple faithfulness check (numeric agreement between the final number in <think> and <answer>) shows 0.39% match rate, indicating the model is not always internally consistent between reasoning trace and final answer. This suggests that while CoT decomposition helps the model structure its approach, reasoning quality remains inconsistent, with many responses showing sound intermediate reasoning but incorrect final answers.

[QUESTION 2] How much does Self-Consistency (K=5) with CoT improve over direct prompting?
--------------------------------------------------------------------------------
With self-consistency (K=5) 

,Method,Num Examples,Format Acc,Answer Acc,Faithfulness,Tie Rate,Avg Top-Vote
0,Direct single-shot,256,31.64%,1.56%,NaN,NaN,NaN
1,CoT single-shot,512,47.07%,19.73%,0.39%,NaN,NaN
2,CoT self-consistency (K=5),256,13.28%,7.42%,NaN,16.02%,54.22%


## GRPO Training

Train Qwen2.5-Math-1.5B with GRPO-Clip on GSM8K for 5 steps.
Run all cells in order: setup → model → hyperparams → train → plot.

In [ ]:
import gc
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# ── Resolve repo root so alignment package is importable ──────────────────
def _find_repo_root() -> Path | None:
    candidates = [
        Path.cwd(), *Path.cwd().parents,
        Path('/content/hw2'),
        Path('/Users/kaywei/Documents/School/cs148b/hw2'),
    ]
    for c in candidates:
        if (c / 'alignment' / 'grpo.py').exists():
            return c
    return None

repo_root = _find_repo_root()
assert repo_root is not None, 'Could not find hw2 repo root'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from alignment.grpo import train_grpo
from alignment.prompts import COT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn

SEED = 0
set_seed(SEED)
random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

In [ ]:
# ── Load GSM8K train / validation splits ──────────────────────────────────
def load_gsm8k(split: str) -> list[dict]:
    ds = load_dataset('openai/gsm8k', 'main', split=split)
    return [{'question': ex['question'], 'answer': ex['answer']} for ex in ds]

train_examples = load_gsm8k('train')
val_examples   = load_gsm8k('test')
print(f'train: {len(train_examples)}  val: {len(val_examples)}')

In [ ]:
# ── Load HuggingFace model + tokenizer ────────────────────────────────────
MODEL_NAME = 'Qwen/Qwen2.5-Math-1.5B'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

policy = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).to(DEVICE)
policy.train()

print('model loaded:', MODEL_NAME)
n_params = sum(p.numel() for p in policy.parameters()) / 1e6
print(f'parameters: {n_params:.0f}M')

In [ ]:
# ── Hyperparameters (from the assignment spec) ────────────────────────────
n_grpo_steps              = 5      # assignment: train for 5 iterations
learning_rate             = 1e-5
advantage_eps             = 1e-6
rollout_batch_size        = 32
group_size                = 8
sampling_temperature      = 1.0
sampling_min_tokens       = 4
sampling_max_tokens       = 256
epochs_per_rollout_batch  = 1      # on-policy
train_batch_size          = 32     # on-policy (== rollout_batch_size)
gradient_accumulation_steps = 16
cliprange                 = 1.0
val_interval              = 1      # validate every step (only 5 total)
val_size                  = 256

assert train_batch_size % gradient_accumulation_steps == 0
assert rollout_batch_size % group_size == 0
assert train_batch_size >= group_size
micro_train_batch_size = train_batch_size // gradient_accumulation_steps
print(f'micro_train_batch_size: {micro_train_batch_size}')

optimizer = torch.optim.Adam(
    policy.parameters(),
    lr=learning_rate,
    weight_decay=0.0,
    betas=(0.9, 0.95),
)
print('optimizer ready')

In [ ]:
# ── Run GRPO training ─────────────────────────────────────────────────────
OUTPUT_DIR = Path('outputs/grpo_training')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results = train_grpo(
    policy=policy,
    tokenizer=tokenizer,
    train_examples=train_examples,
    val_examples=val_examples,
    reward_fn=answer_tag_reward_fn,
    optimizer=optimizer,
    device=DEVICE,
    prompt_template=str(COT_PROMPT_TEMPLATE),
    n_grpo_steps=n_grpo_steps,
    rollout_batch_size=rollout_batch_size,
    group_size=group_size,
    sampling_temperature=sampling_temperature,
    sampling_min_tokens=sampling_min_tokens,
    sampling_max_tokens=sampling_max_tokens,
    epochs_per_rollout_batch=epochs_per_rollout_batch,
    train_batch_size=train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    cliprange=cliprange,
    advantage_eps=advantage_eps,
    normalize_by_std=True,
    val_interval=val_interval,
    val_size=val_size,
    val_sampling_max_tokens=256,
    output_dir=OUTPUT_DIR,
    seed=SEED,
)

# Save full results
with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Training complete. Results saved to', OUTPUT_DIR / 'results.json')

In [ ]:
# ── Plot training + validation rewards ────────────────────────────────────
history     = results['history']
val_history = results['val_history']

train_steps   = [h['step'] for h in history]
train_rewards = [h['mean_reward'] for h in history]
train_answers = [h['mean_answer_reward'] for h in history]
train_losses  = [h['loss'] for h in history]

val_steps   = [v['step'] for v in val_history]
val_rewards = [v['val_reward'] for v in val_history]
val_answers = [v['val_answer_reward'] for v in val_history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_steps, train_rewards, 'o-', label='train total')
axes[0].plot(train_steps, train_answers, 's--', label='train answer')
axes[0].plot(val_steps, val_rewards, 'o-', label='val total')
axes[0].plot(val_steps, val_answers, 's--', label='val answer')
axes[0].set_xlabel('GRPO step')
axes[0].set_ylabel('reward')
axes[0].set_title('Reward over GRPO steps')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(val_steps, val_answers, 'o-', color='green')
axes[1].set_xlabel('GRPO step')
axes[1].set_ylabel('val answer reward')
axes[1].set_title('Validation Answer Reward')
axes[1].grid(True)

axes[2].plot(train_steps, train_losses, 'o-', color='red')
axes[2].set_xlabel('GRPO step')
axes[2].set_ylabel('loss')
axes[2].set_title('Training Loss')
axes[2].grid(True)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()
print('Plot saved to', OUTPUT_DIR / 'training_curves.png')

In [ ]:
# ── Show example rollouts from the final trained policy ───────────────────
from alignment.rewards import extract_answer_from_tags

N_EXAMPLES = 4
sample_examples = val_examples[:N_EXAMPLES]
sample_prompts  = [str(COT_PROMPT_TEMPLATE).format(question=ex['question']) for ex in sample_examples]
sample_gts      = [ex['answer'] for ex in sample_examples]

# Use greedy decoding for deterministic examples
tokenizer.padding_side = 'left'
enc = tokenizer(sample_prompts, return_tensors='pt', padding=True, add_special_tokens=False)
input_ids = enc['input_ids'].to(DEVICE)
attention_mask = enc['attention_mask'].to(DEVICE)
prompt_len = input_ids.shape[1]

policy.eval()
with torch.no_grad():
    out = policy.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        do_sample=False,
        max_new_tokens=256,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
policy.train()

for i, (ex, ids) in enumerate(zip(sample_examples, out)):
    response = tokenizer.decode(ids[prompt_len:], skip_special_tokens=True)
    if '</answer>' in response:
        response = response[:response.index('</answer>') + len('</answer>')]
    score = answer_tag_reward_fn(response, ex['answer'])
    pred  = extract_answer_from_tags(response)
    print(f'--- Example {i+1} ---')
    print(f'Question : {ex["question"][:120]}')
    print(f'Response : {response[:300]}')
    print(f'Predicted: {pred}  |  Correct: {score["answer_reward"]==1.0}')
    print()